
# Qwen3 Next — from scratch

Welcome! This notebook rebuilds the core pieces of **Qwen 3 Next** step by step in the same spirit as `naklecha/llama3-from-scratch`. We will dissect the architectural choices that make Qwen 3 Next stand out: the Gated DeltaNet state-space module, partial rotary position encodings, and the multi-token prediction head that upgrades autoregressive training. Along the way we sprinkle interactive widgets, visual diagnostics, and commentary about how these ideas compare with earlier transformers and Mamba-style state-space models.



## Notebook map
1. Quick references and environment prep
2. Configuration and helper utilities
3. Partial Rotary Position Encoding (RoPE)
4. Gated DeltaNet: the new mixing block
5. Causal attention with partial RoPE and grouped-query heads
6. Multi-token prediction (MTP) head
7. Wiring everything into `Qwen3NextForCausalLM`
8. Interactive experiments and comparisons with Mamba
9. Lightweight sanity checks and generation demo



> **Reading order tip**: Cells marked with ✏️ expose implementation details. Cells marked with 🧪 help you poke at the model. Feel free to skip longer math blocks on a first read—the code comes with comments that highlight the important bits.



## Quick references
- Qwen 3 technical report (Songlin Yang et al., 2025) — introduces Gated DeltaNet and multi-token prediction.
- Gated DeltaNet paper — dives into the selective scanning mechanism.
- Mamba (Gu et al., 2023) — state space models with selective scan; we will compare against it later.
- `naklecha/llama3-from-scratch` — the tutorial that inspired the narrative structure here.



## Spotlight: Qwen3-Next-80B-A3B
The flagship release in the Qwen3-Next family is the **80B-A3B** variant (80 billion total parameters, ~3 billion activated per token). Its public model card highlights several architectural upgrades:
- Hybrid attention stack that alternates Gated DeltaNet and **Gated Attention** blocks, targeting 256K+ context windows.
- Sparse MoE layers with 512 experts (10 active + 1 shared) to keep FLOPs manageable.
- Stability tweaks: zero-centered/decayed layer norms, upgraded RMSNorm, and long-context training tricks.
- Native multi-token prediction and speculative decoding hooks (NEXTN/EAGLE) baked into the inference recipe.

Below you will find a distilled spec sheet. Use it as a reference when adapting the toy config to the production-scale layout.


In [ ]:

qwen80b_a3b_spec = {
    "total_params": 80_000_000_000,
    "activated_params": 3_000_000_000,
    "layers": 48,
    "hidden_size": 2048,
    "gated_attention": {
        "q_heads": 16,
        "kv_heads": 2,
        "head_dim": 256,
        "rope_dim": 64,
    },
    "gated_deltanet": {
        "v_heads": 32,
        "qk_heads": 16,
        "head_dim": 128,
    },
    "moe": {
        "experts": 512,
        "active_experts": 10,
        "shared_experts": 1,
        "intermediate_dim": 512,
    },
    "context": {
        "native": 262_144,
        "max_tested": 1_010_000,
    },
}

qwen80b_a3b_spec  # Feel free to pretty-print or compare against toy configs.


In [ ]:

# If you run this notebook outside the Codex CLI environment, you may need to install dependencies.
# !pip install -q torch torchvision torchaudio ipywidgets matplotlib


In [ ]:

import math
from dataclasses import dataclass
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from IPython.display import display
    import ipywidgets as widgets
    _widgets_available = True
except ModuleNotFoundError:
    _widgets_available = False

print(f"PyTorch {torch.__version__} on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
DEFAULT_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_float32_matmul_precision('high')



## Configuration dataclasses
Qwen 3 Next keeps the familiar LLaMA-style configuration fields but adds twists:
- `partial_rotary_factor` decides how many head dimensions receive rotary embeddings (Qwen uses ~50%).
- `gated_deltanet_hidden` controls the internal state width of the DeltaNet block.
- `multi_token_predictions` chooses how many future tokens we predict per position. A value of 1 reduces to classical next-token training.


In [ ]:

@dataclass
class QwenConfig:
    vocab_size: int = 151936
    hidden_size: int = 1024
    num_hidden_layers: int = 24
    num_attention_heads: int = 16
    num_key_value_heads: int = 8  # Grouped-query attention
    intermediate_size: int = 4096
    rms_norm_eps: float = 1e-6
    partial_rotary_factor: float = 0.5
    rope_theta: float = 1000000.0
    max_position_embeddings: int = 65536
    gated_deltanet_hidden: int = 3072
    gated_deltanet_kernel: int = 4
    multi_token_predictions: int = 4
    multi_token_weight: float = 0.5  # Loss interpolation weight
    use_bias: bool = False
    device: str = DEFAULT_DEVICE
    dtype: torch.dtype = torch.float32

    @property
    def head_dim(self) -> int:
        return self.hidden_size // self.num_attention_heads

    @property
    def kv_dim(self) -> int:
        return self.hidden_size // self.num_attention_heads * self.num_key_value_heads

    def to_kwargs(self):
        return {
            'device': self.device,
            'dtype': self.dtype,
        }



## Utility layers
Qwen sticks with RMSNorm and uses a dropout-free design (dropout is replaced by data augmentation and MTP). We also add helper functions for parameter initialization.


In [ ]:

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed = x * torch.rsqrt(torch.mean(x.pow(2), dim=-1, keepdim=True) + self.eps)
        return normed * self.weight


def init_linear(linear: nn.Linear, fan_in: int) -> None:
    nn.init.kaiming_uniform_(linear.weight, a=math.sqrt(5))
    if linear.bias is not None:
        bound = 1 / math.sqrt(fan_in)
        nn.init.uniform_(linear.bias, -bound, bound)



# 1. Partial Rotary Position Encoding (RoPE)
Rotary embeddings rotate query/key pairs in the complex plane. In Qwen 3 Next only a subset of head dimensions get rotated; the rest stay static, letting the model keep a pure content subspace. The `partial_rotary_factor` describes that split.



### Intuition
Let `d` be the head dimension. Classical RoPE rotates all `d/2` complex pairs. Partial RoPE leaves `(1 - factor) * d` untouched:
- The rotated slice keeps long-range inductive bias.
- The unrotated slice acts like a learned absolute embedding.
This balance is important for multi-token prediction, which thrives on precise local modeling.


In [ ]:

class PartialRotaryEmbedding:
    def __init__(self, head_dim: int, max_position: int, base_theta: float, fraction: float, device: str, dtype: torch.dtype):
        self.head_dim = head_dim
        self.rotary_dim = int(head_dim * fraction)
        self.rotary_dim -= self.rotary_dim % 2  # even required for complex pairs
        self.scale = base_theta
        self.max_position = max_position
        self.device = device
        self.dtype = dtype
        self._build_cache(max_position)

    def _build_cache(self, length: int) -> None:
        if self.rotary_dim == 0:
            self.cos_cached = None
            self.sin_cached = None
            return
        theta = self.scale ** (-torch.arange(0, self.rotary_dim, 2, device=self.device, dtype=self.dtype) / self.rotary_dim)
        positions = torch.arange(length, device=self.device, dtype=self.dtype)
        angles = positions[:, None] * theta[None, :]
        self.cos_cached = torch.cos(angles)
        self.sin_cached = torch.sin(angles)

    def __call__(self, positions: torch.Tensor) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
        if self.rotary_dim == 0:
            return None, None
        if positions.max().item() >= self.cos_cached.size(0):
            self._build_cache(int(positions.max().item()) + 128)
        return self.cos_cached.index_select(0, positions), self.sin_cached.index_select(0, positions)


def apply_partial_rotary(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, rotary_dim: int) -> torch.Tensor:
    if rotary_dim == 0:
        return x
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    x_rot, x_pass = x[..., :rotary_dim], x[..., rotary_dim:]
    x_rot = x_rot.reshape(*x_rot.shape[:-1], rotary_dim // 2, 2)
    x_rotated = torch.empty_like(x_rot)
    x_rotated[..., 0] = x_rot[..., 0] * cos - x_rot[..., 1] * sin
    x_rotated[..., 1] = x_rot[..., 0] * sin + x_rot[..., 1] * cos
    x_rotated = x_rotated.reshape(*x_rotated.shape[:-2], rotary_dim)
    return torch.cat([x_rotated, x_pass], dim=-1)



🧪 **Interactive** — explore the effect of different partial factors on the rotation magnitude. The plot tracks the norm change induced by RoPE on a random head.


In [ ]:

if _widgets_available:
    import matplotlib.pyplot as plt

    def visualize_partial_rope(fraction: float = 0.5):
        head_dim = 64
        positions = torch.arange(128)
        rope = PartialRotaryEmbedding(head_dim, 2048, base_theta=1_000_000, fraction=fraction, device='cpu', dtype=torch.float32)
        cos, sin = rope(positions)
        x = torch.randn(1, 4, positions.size(0), head_dim)
        rotated = apply_partial_rotary(x, cos, sin, rope.rotary_dim)
        norms_before = x.norm(dim=-1).mean(dim=(0, 1)).detach().cpu().numpy()
        norms_after = rotated.norm(dim=-1).mean(dim=(0, 1)).detach().cpu().numpy()
        plt.figure(figsize=(6, 3))
        plt.plot(norms_before, label='before')
        plt.plot(norms_after, label='after')
        plt.title(f'Partial RoPE fraction={fraction:.2f}')
        plt.xlabel('Position')
        plt.ylabel('Mean L2 norm')
        plt.legend()
        plt.show()

    widgets.interact(visualize_partial_rope, fraction=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.5));
else:
    print('ipywidgets not available; install ipywidgets to enable the interactive chart.')



# 2. Gated DeltaNet block
Qwen 3 Next replaces the classical feed-forward network with **Gated DeltaNet**. The block injects state-space style sequential bias while keeping transformer data flow. Conceptually:
1. Project the input into three streams: a residual proposal, a delta update, and a gate controller.
2. Run the delta stream through a depthwise convolution followed by a cumulative integration (DeltaNet).
3. Mix the integrated state back into the residual with a learned gate.
This marries the long-range context of SSMs with transformer residuals.



### Implementation notes
- The delta stream uses a SiLU activation and depthwise conv (kernel size 4 by default) to capture local derivatives.
- We integrate the deltas with `torch.cumsum`, mirroring the additive scan described in the Gated DeltaNet paper.
- The gate is sigmoid-controlled, allowing the block to revert to identity when necessary.
- A final projection matches the hidden size so the residual branch stays dimensionally consistent.


In [ ]:

class GatedDeltaNet(nn.Module):
    def __init__(self, config: QwenConfig):
        super().__init__()
        dim = config.hidden_size
        hidden = config.gated_deltanet_hidden
        kernel = config.gated_deltanet_kernel
        self.input_proj = nn.Linear(dim, dim * 3, bias=config.use_bias)
        self.delta_proj = nn.Linear(dim, hidden, bias=config.use_bias)
        self.depthwise_conv = nn.Conv1d(hidden, hidden, kernel, groups=hidden, padding=kernel - 1)
        self.output_proj = nn.Linear(hidden, dim, bias=config.use_bias)
        self.residual_proj = nn.Linear(dim, dim, bias=config.use_bias)
        self.norm = RMSNorm(dim, config.rms_norm_eps)
        self.dropout = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shortcut = x
        x = self.norm(x)
        gate_raw, proposal, delta_tokens = self.input_proj(x).chunk(3, dim=-1)
        gate = torch.sigmoid(gate_raw)
        residual_part = self.residual_proj(proposal)

        delta_state = F.silu(self.delta_proj(delta_tokens))
        delta_state = self.depthwise_conv(delta_state.transpose(1, 2))
        delta_state = delta_state[:, :, :shortcut.size(1)].transpose(1, 2)
        delta_integrated = torch.cumsum(delta_state, dim=1)
        mixed = gate * self.output_proj(delta_integrated) + (1 - gate) * residual_part
        return shortcut + self.dropout(mixed)



🧪 **Interactive** — move the gate towards 0 or 1 to see how the block interpolates between residual and delta paths.


In [ ]:

if _widgets_available:
    def gated_delta_probe(gate_bias: float = 0.0):
        cfg = QwenConfig(hidden_size=64, gated_deltanet_hidden=128, use_bias=True)
        block = GatedDeltaNet(cfg)
        with torch.no_grad():
            block.input_proj.bias[:cfg.hidden_size].fill_(gate_bias)
        x = torch.randn(2, 32, cfg.hidden_size)
        out = block(x)
        diff = (out - x).norm(dim=-1).mean().item()
        print(f'Average update magnitude: {diff:.4f}')

    widgets.interact(gated_delta_probe, gate_bias=widgets.FloatSlider(min=-5.0, max=5.0, step=0.5, value=0.0));
else:
    print('ipywidgets not available; install ipywidgets to enable the interactive gate demo.')



# 3. Causal attention with grouped keys
Self-attention stays close to LLaMA-family implementations but we highlight two Qwen-specific tweaks:
- **Grouped-query attention (GQA)** lowers KV memory by sharing key/value projections across head groups.
- **Partial RoPE** slots directly into the query/key projection before scoring.


In [ ]:

class Attention(nn.Module):
    def __init__(self, config: QwenConfig):
        super().__init__()
        self.config = config
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.head_dim

        self.scale = self.head_dim ** -0.5
        self.q_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=config.use_bias)
        self.k_proj = nn.Linear(config.hidden_size, config.kv_dim, bias=config.use_bias)
        self.v_proj = nn.Linear(config.hidden_size, config.kv_dim, bias=config.use_bias)
        self.o_proj = nn.Linear(config.hidden_size, config.hidden_size, bias=config.use_bias)

        self.rotary = PartialRotaryEmbedding(
            head_dim=config.head_dim,
            max_position=config.max_position_embeddings,
            base_theta=config.rope_theta,
            fraction=config.partial_rotary_factor,
            device=config.device,
            dtype=config.dtype,
        )
        self.rotary_dim = self.rotary.rotary_dim
        self.norm = RMSNorm(config.hidden_size, config.rms_norm_eps)

    def reshape_heads(self, x: torch.Tensor, num_heads: int) -> torch.Tensor:
        return x.view(x.size(0), x.size(1), num_heads, self.head_dim).transpose(1, 2)

    def repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        if self.num_heads == self.num_kv_heads:
            return x
        repeat = self.num_heads // self.num_kv_heads
        return x.repeat_interleave(repeat, dim=1)

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, positions: Optional[torch.Tensor] = None) -> torch.Tensor:
        shortcut = x
        x = self.norm(x)
        q = self.reshape_heads(self.q_proj(x), self.num_heads)
        k = self.reshape_heads(self.k_proj(x), self.num_kv_heads)
        v = self.reshape_heads(self.v_proj(x), self.num_kv_heads)

        if positions is None:
            positions = torch.arange(x.size(1), device=x.device)
        cos, sin = self.rotary(positions)
        if cos is not None:
            q = apply_partial_rotary(q, cos, sin, self.rotary_dim)
            k = apply_partial_rotary(k, cos, sin, self.rotary_dim)

        k = self.repeat_kv(k)
        v = self.repeat_kv(v)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        seq_len = q.size(-2)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=q.device, dtype=torch.bool), diagonal=1)
        attn_scores = attn_scores.masked_fill(causal_mask, float('-inf'))
        if attention_mask is not None:
            attn_scores += attention_mask
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v)
        attn_output = attn_output.transpose(1, 2).contiguous().view(x.size(0), x.size(1), -1)
        return shortcut + self.o_proj(attn_output)



# 4. Transformer block and stack
Each layer chains attention and Gated DeltaNet, with RMSNorm applied pre-activation. We also expose a cache-friendly forward for incremental generation.


In [ ]:

class QwenBlock(nn.Module):
    def __init__(self, config: QwenConfig):
        super().__init__()
        self.attention = Attention(config)
        self.mlp = GatedDeltaNet(config)

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, positions: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = self.attention(x, attention_mask=attention_mask, positions=positions)
        x = self.mlp(x)
        return x


class QwenModel(nn.Module):
    def __init__(self, config: QwenConfig):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([QwenBlock(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, config.rms_norm_eps)

    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        hidden = self.embed_tokens(input_ids)
        for layer in self.layers:
            hidden = layer(hidden, attention_mask=attention_mask, positions=positions)
        return self.norm(hidden)



# 5. Multi-token prediction head
Qwen 3 Next augments next-token prediction with several additional offsets. For each timestep we predict the token at `t + k` for `k in {1, ..., K}`. The official model uses a learned interpolation weight to blend classical loss and future-token losses.



We implement the head as `K` parallel linear projections. During training we shift the hidden states accordingly and compute a weighted cross-entropy.


In [ ]:

class MultiTokenPredictionHead(nn.Module):
    def __init__(self, config: QwenConfig):
        super().__init__()
        self.config = config
        self.projections = nn.ModuleList([
            nn.Linear(config.hidden_size, config.vocab_size, bias=False)
            for _ in range(config.multi_token_predictions)
        ])

    def tie_weights(self, embedding: nn.Embedding):
        for proj in self.projections:
            proj.weight = embedding.weight

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        logits = [proj(hidden) for proj in self.projections]
        return torch.stack(logits, dim=1)


def multi_token_cross_entropy(logits: torch.Tensor, input_ids: torch.Tensor, ignore_index: int = -100, multi_token_weight: float = 0.5) -> torch.Tensor:
    batch, predictions, seq_len, vocab = logits.shape
    losses = []
    for offset in range(predictions):
        shifted = torch.roll(input_ids, shifts=-(offset + 1), dims=1)
        shifted[:, - (offset + 1):] = ignore_index
        flat_logits = logits[:, offset].reshape(batch * seq_len, vocab)
        flat_targets = shifted.reshape(batch * seq_len)
        loss = F.cross_entropy(flat_logits, flat_targets, ignore_index=ignore_index)
        losses.append(loss)
    main_loss = losses[0]
    if predictions == 1:
        return main_loss
    aux = torch.stack(losses[1:]).mean()
    return main_loss * (1 - multi_token_weight) + aux * multi_token_weight



# 6. Full language model wrapper
We now bind embeddings, transformer stack, and MTP head. The `generate` helper performs greedy decoding and highlights how multi-token predictions can be used for speculative lookahead.


In [ ]:

class Qwen3NextForCausalLM(nn.Module):
    def __init__(self, config: QwenConfig):
        super().__init__()
        self.config = config
        self.model = QwenModel(config)
        self.lm_head = MultiTokenPredictionHead(config)
        self.lm_head.tie_weights(self.model.embed_tokens)

    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None):
        hidden = self.model(input_ids, attention_mask=attention_mask)
        logits = self.lm_head(hidden)
        return logits

    @torch.no_grad()
    def generate(self, input_ids: torch.Tensor, max_new_tokens: int = 20) -> torch.Tensor:
        self.eval()
        generated = input_ids.clone()
        for _ in range(max_new_tokens):
            logits = self.forward(generated)
            next_token_logits = logits[:, 0, -1]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
        return generated



# 7. Parameter sanity checks
Instantiate a toy configuration to verify tensor shapes and loss wiring. We keep vocab small so the notebook runs quickly on CPU.


In [ ]:

toy_cfg = QwenConfig(
    vocab_size=128,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=8,
    num_key_value_heads=4,
    intermediate_size=512,
    gated_deltanet_hidden=256,
    multi_token_predictions=3,
    multi_token_weight=0.4,
    device='cpu',
    dtype=torch.float32,
)
model = Qwen3NextForCausalLM(toy_cfg)
inputs = torch.randint(0, toy_cfg.vocab_size, (2, 16))
logits = model(inputs)
print('Logits shape:', logits.shape)
loss = multi_token_cross_entropy(logits, inputs, multi_token_weight=toy_cfg.multi_token_weight)
print('Sample loss:', float(loss))



# 8. Comparing Gated DeltaNet to Mamba
Both Gated DeltaNet and Mamba belong to the **selective scan** family, but they slot into transformers differently.
- Mamba replaces attention with a continuous-time state space layer. It excels at long sequences with linear compute.
- Qwen keeps attention but swaps the MLP for a gated scan module. This preserves softmax expressivity while adding a controllable recurrence path.

Below we sketch a light Mamba-style block and compare update magnitudes.


In [ ]:

class TinyMambaBlock(nn.Module):
    def __init__(self, dim: int, state: int):
        super().__init__()
        self.input_proj = nn.Linear(dim, 2 * state)
        self.state_proj = nn.Linear(state, dim)
        self.conv = nn.Conv1d(state, state, kernel_size=3, padding=2, groups=state)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        u, v = self.input_proj(x).chunk(2, dim=-1)
        u = torch.sigmoid(u)
        v = F.silu(v)
        v = self.conv(v.transpose(1, 2))[:, :, :t].transpose(1, 2)
        scanned = torch.cumsum(v, dim=1)
        return x + self.state_proj(u * scanned)


def compare_blocks():
    x = torch.randn(2, 64, 256)
    delta_block = GatedDeltaNet(QwenConfig(hidden_size=256, gated_deltanet_hidden=384))
    mamba_block = TinyMambaBlock(dim=256, state=192)
    delta_update = (delta_block(x) - x).norm(dim=-1).mean().item()
    mamba_update = (mamba_block(x) - x).norm(dim=-1).mean().item()
    print(f'Gated DeltaNet update norm: {delta_update:.4f}')
    print(f'Tiny Mamba update norm:    {mamba_update:.4f}')

compare_blocks()



# 9. Lightweight training loop (optional)
The following cell shows how you might fine-tune the toy model with multi-token loss. It trains on random data by default; plug in your dataset iterator for real experiments.


In [ ]:

@torch.no_grad()
def token_accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = logits[:, 0].argmax(dim=-1)
    correct = (preds == targets).float()
    mask = targets != -100
    return (correct * mask).sum().div(mask.sum().clamp_min(1.0)).item()


def train_toy(model: nn.Module, steps: int = 20, lr: float = 3e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95))
    for step in range(steps):
        batch = torch.randint(0, model.config.vocab_size, (4, 32))
        logits = model(batch)
        loss = multi_token_cross_entropy(logits, batch, multi_token_weight=model.config.multi_token_weight)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if step % 5 == 0:
            acc = token_accuracy(logits, batch)
            print(f'step {step:02d} | loss {loss.item():.4f} | acc {acc:.4f}')

train_toy(model)



# 10. Greedy generation demo
Even the toy model can produce sequences (they will look random because the model is random, but the wiring works).


In [ ]:

prompt = torch.randint(0, toy_cfg.vocab_size, (1, 8))
completion = model.generate(prompt, max_new_tokens=10)
print('Prompt:', prompt.tolist())
print('Completion:', completion.tolist())



## Where to go next
- Swap `toy_cfg` for real Qwen 3 Next hyperparameters (7B: 32 layers, 4096 hidden, 14336 Delta hidden, etc.).
- Load tokenizer weights from the official release and plug them into the embedding + tied head.
- Port the selective scan to a custom CUDA kernel for training speed (the official repo uses Triton).
- Experiment with `multi_token_predictions` > 4; Gated DeltaNet's stabilizing recurrence helps the model digest extra supervision.

Happy hacking! 💫
